## Below is ALL ALE (Accumulated Local Effects) code in ONE SINGLE NOTEBOOK CELL.
You can copy–paste this entire cell and run it once.

It covers:

✅ 1D ALE (single feature)

✅ Multiple 1D ALEs

✅ 2D ALE (feature interaction)

✅ Clean, publication-style plots

In [ ]:
# ============================================================
# Accumulated Local Effects (ALE) — FULL ONE-CELL SCRIPT
# ============================================================

# (1) Install dependency (run once in Colab / fresh env)
!pip -q install pyALE

# (2) Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from pyALE import ale

sns.set(style="whitegrid")
np.random.seed(42)

# ------------------------------------------------------------
# (3) Dataset
# ------------------------------------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ------------------------------------------------------------
# (4) Train Model
# ------------------------------------------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ------------------------------------------------------------
# (5) ALE – Single Feature (1D)
# ------------------------------------------------------------
plt.figure(figsize=(6, 4))
ale(
    X=X_train,
    model=model,
    feature=["mean radius"],
    grid_size=40
)
plt.title("ALE Plot – mean radius")
plt.show()

# ------------------------------------------------------------
# (6) ALE – Multiple Features (1D)
# ------------------------------------------------------------
features_1d = ["mean radius", "mean texture", "mean concavity"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, features_1d):
    ale(
        X=X_train,
        model=model,
        feature=[feat],
        grid_size=40,
        ax=ax
    )
    ax.set_title(f"ALE – {feat}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# (7) ALE – Top Features Automatically
# ------------------------------------------------------------
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

top_feats = importances.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top_feats):
    ale(
        X=X_train,
        model=model,
        feature=[feat],
        grid_size=40,
        ax=ax
    )
    ax.set_title(f"ALE – {feat}")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# (8) ALE – 2D Interaction Plot
# ------------------------------------------------------------
plt.figure(figsize=(6, 5))
ale(
    X=X_train,
    model=model,
    feature=["mean radius", "mean concavity"],
    grid_size=30
)
plt.title("2D ALE – Interaction Effect")
plt.show()

# ------------------------------------------------------------
# (9) ALE vs PDP intuition note
# ------------------------------------------------------------
print("""
ALE properties:
✔ Unbiased under correlated features
✔ Local differences → accumulated globally
✔ Preferred over PDP when multicollinearity exists
""")
